# 04 · Own approach: decisions for macro-F1, blending and stacking

Three experiments on the saved class probabilities of notebooks 02 and 03, each compared with one fixed reference:
XLM-R with the mean over all chunks (0.899 test-like macro-F1 on validation, 0.893 on the public leaderboard).

1. **Class weights.** A model predicts the class with the highest probability. Macro-F1 gives the three classes
   equal weight, so it can pay off to multiply the probabilities by per-class factors before taking the maximum.
   XLM-R finds 91% of fusion texts but only 77% of human texts, so the factor for human should be above 1.
2. **Blend.** A weighted average of the XLM-R and TF-IDF probabilities. The two models see different things
   (context vs exact character sequences), so their errors may partly cancel.
3. **Stacking.** A small logistic regression that takes the probabilities of both models, the first-chunk
   probabilities, the number of chunks and the domain, and learns how to combine them; the domain is known for
   test texts too.

All three are fitted on the validation set, so they cannot be scored on the rows they were fitted on. Validation is
split into 5 group folds (the same source groups as before); each fold is predicted by a version fitted on the
other four (*out-of-fold*). Only these out-of-fold scores are compared. For test, the final version is fitted on
the whole validation set. Both base models never saw the validation texts in training.

In [1]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

platform: local
    data: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data
 outputs: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs
   cache: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/cache
    repo: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection
  device: mps
data ready at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data


PosixPath('/Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data')

In [2]:
# Imports, config; validation labels, groups and weights; saved probabilities of both models, aligned by id
import time
import numpy as np
import pandas as pd
from itertools import product
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder

from hwkit.config import load_config
from hwkit.split import LABELS, load_split, scores
from hwkit.submission import make_submission

NOTEBOOK_START = time.time()
cfg = load_config("approach")
METRICS = ["macro_f1_testlike", "macro_f1", "accuracy", "f1_human", "f1_fusion", "f1_ai"]
COLS = [f"s_{c}" for c in LABELS]

cols = ["id", "label", "domain", "lang"]
val = load_split(pd.read_csv(P.data / "train.csv", usecols=cols))
val = val[val["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv", usecols=["id", "domain", "lang"])
for df in (val, test):
    df["domain_lang"] = df["domain"] + " | " + df["lang"]

def load_probs(run_id, split, ids):
    """Probabilities of a saved run in LABELS order, rows in the order of `ids`."""
    pr = pd.read_csv(P.runs / run_id / "predictions" / f"{split}.csv").set_index("id")
    return pr.loc[ids, COLS].to_numpy(dtype=np.float64)

def n_chunks(split, ids):
    """Number of 512-token chunks per text, from the chunk file of the encoder run."""
    order = pd.read_csv(P.runs / cfg["inputs"]["encoder"] / "predictions" / f"{split}.csv", usecols=["id"])["id"]
    counts = np.bincount(np.load(P.runs / cfg["inputs"]["encoder"] / "predictions" / f"chunks_{split}.npz")["doc"])
    return pd.Series(counts, index=order).loc[ids].to_numpy()

prob = {}
for split, df in (("val", val), ("test", test)):
    prob[split] = {"xlmr": load_probs(cfg["inputs"]["encoder"], split, df["id"]),
                   "xlmr_first": load_probs(cfg["inputs"]["encoder"], f"{split}_first", df["id"]),
                   "tfidf": load_probs(cfg["inputs"]["baseline"], split, df["id"])}
    df["n_chunks"] = n_chunks(split, df["id"])
y = val["label"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()
w = val["weight"].to_numpy()
run = Run("approach", cfg, P)
print(f"validation: {len(val)} texts, test: {len(test)}")

run 'approach' at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/runs/approach  done: nothing yet
validation: 27236 texts, test: 25000


In [3]:
# Out-of-fold machinery: 5 group folds inside validation, fast weighted macro-F1 for the searches
folds = np.zeros(len(val), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=cfg["cv"]["folds"], shuffle=True, random_state=42)
for k, (_, idx) in enumerate(sgkf.split(val, val["label"], val["group"])):
    folds[idx] = k

def macro_f1(y_true, pred, weight):
    """Weighted macro-F1 over the three classes (same value as sklearn, much faster inside grid searches)."""
    f1 = []
    for c in range(len(LABELS)):
        tp = weight[(y_true == c) & (pred == c)].sum()
        wrong = weight[(y_true == c) != (pred == c)].sum()          # false positives + false negatives
        f1.append(2 * tp / (2 * tp + wrong) if tp + wrong > 0 else 0.0)
    return float(np.mean(f1))

def out_of_fold(fit, apply):
    """fit(rows) -> params on the training folds; apply(params, rows) -> class scores for the held-out fold."""
    oof = np.zeros((len(val), len(LABELS)))
    for k in range(cfg["cv"]["folds"]):
        oof[folds == k] = apply(fit(folds != k), folds == k)
    return oof

def report(name, s, extra=None):
    """Scores of a variant (out-of-fold class scores s) as one table row, also logged in the run."""
    res = scores(val["label"], np.array(LABELS)[s.argmax(axis=1)], val["weight"])
    run.log_metrics(name, {**res, **(extra or {})})
    return {"variant": name, **{k: res[k] for k in METRICS}}

split_groups = int((pd.Series(folds).groupby(val["group"]).nunique() > 1).sum())
print("texts per fold:", np.bincount(folds).tolist(), "| groups split between folds:", split_groups)

texts per fold: [5448, 5448, 5446, 5447, 5447] | groups split between folds: 0


## Reference and the plain models

In [4]:
# The fixed reference (XLM-R, mean of chunks) and the other untuned predictions on the same rows
rows = [report("tfidf", prob["val"]["tfidf"]), report("xlmr_first_chunk", prob["val"]["xlmr_first"]),
        report("reference: xlmr", prob["val"]["xlmr"])]
pd.DataFrame(rows).round(4)

,variant,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai
0,tfidf,0.8657,0.8717,0.8917,0.8081,0.8219,0.9850
1,xlmr_first_chunk,0.8599,0.8577,0.8818,0.7743,0.8127,0.9861
2,reference: xlmr,0.8985,0.8975,0.9158,0.8287,0.8739,0.9900


## Experiment 1: class weights for macro-F1

In [5]:
# Grid search of the human and fusion multipliers (ai = 1) on the training folds, applied to the held-out fold
g = cfg["weights_grid"]
GRID = np.exp(np.linspace(g["low"], g["high"], g["steps"]))

def fit_weights(p, rows):
    best = (-1.0, None)
    for wh, wf in product(GRID, GRID):
        m = np.array([wh, wf, 1.0])
        score = macro_f1(y[rows], (p[rows] * m).argmax(axis=1), w[rows])
        if score > best[0]:
            best = (score, m)
    return best[1]

p_ref = prob["val"]["xlmr"]
oof_weights = out_of_fold(lambda rows: fit_weights(p_ref, rows), lambda m, rows: p_ref[rows] * m)
m_full = fit_weights(p_ref, np.ones(len(val), dtype=bool))
rows.append(report("xlmr + class weights", oof_weights, {"w_human": m_full[0], "w_fusion": m_full[1]}))
print(f"multipliers fitted on all of validation: human {m_full[0]:.2f}, fusion {m_full[1]:.2f}, ai 1")
pd.DataFrame(rows).round(4)

multipliers fitted on all of validation: human 4.48, fusion 4.48, ai 1


,variant,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai
0,tfidf,0.8657,0.8717,0.8917,0.8081,0.8219,0.9850
1,xlmr_first_chunk,0.8599,0.8577,0.8818,0.7743,0.8127,0.9861
2,reference: xlmr,0.8985,0.8975,0.9158,0.8287,0.8739,0.9900
3,xlmr + class weights,0.8993,0.8985,0.9167,0.8293,0.8745,0.9916


**Findings.** Class weights barely help (+0.001). The search gives human and fusion the same factor, so it does not
move the human/fusion boundary at all; it only makes ai a bit less likely, and that factor sits at the edge of the
grid. Shifting the decision towards human, which the low human recall suggested, does not raise macro-F1: fewer human
texts would be missed, but more fusion texts would be called human.

## Experiment 2: blend of XLM-R and TF-IDF

In [6]:
# TF-IDF share a in (1 - a) * XLM-R + a * TF-IDF, chosen on the training folds; then also with class weights on top
SHARES = np.round(np.arange(0, 1 + 1e-9, cfg["blend_grid"]), 2)

def blend(a, split="val"):
    return (1 - a) * prob[split]["xlmr"] + a * prob[split]["tfidf"]

def fit_share(rows):
    return max(SHARES, key=lambda a: macro_f1(y[rows], blend(a)[rows].argmax(axis=1), w[rows]))

def fit_share_weights(rows):
    a = fit_share(rows)
    return a, fit_weights(blend(a), rows)

oof_blend = out_of_fold(fit_share, lambda a, rows: blend(a)[rows])
oof_blend_w = out_of_fold(fit_share_weights, lambda am, rows: blend(am[0])[rows] * am[1])
a_full, m_blend = fit_share_weights(np.ones(len(val), dtype=bool))
rows.append(report("blend", oof_blend, {"tfidf_share": fit_share(np.ones(len(val), dtype=bool))}))
rows.append(report("blend + class weights", oof_blend_w, {"tfidf_share": a_full}))
curve = {a: macro_f1(y, blend(a).argmax(axis=1), w) for a in SHARES}
print(f"TF-IDF share fitted on all of validation: {a_full:.2f}; class multipliers on top: "
      f"human {m_blend[0]:.2f}, fusion {m_blend[1]:.2f}")
print("in-sample test-like macro-F1 by TF-IDF share:", {float(a): round(curve[a], 4) for a in SHARES[::4]})
pd.DataFrame(rows).round(4)

TF-IDF share fitted on all of validation: 0.30; class multipliers on top: human 2.72, fusion 2.40
in-sample test-like macro-F1 by TF-IDF share: {0.0: 0.8985, 0.2: 0.9065, 0.4: 0.9065, 0.6: 0.8947, 0.8: 0.877, 1.0: 0.8657}


,variant,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai
0,tfidf,0.8657,0.8717,0.8917,0.8081,0.8219,0.9850
1,xlmr_first_chunk,0.8599,0.8577,0.8818,0.7743,0.8127,0.9861
2,reference: xlmr,0.8985,0.8975,0.9158,0.8287,0.8739,0.9900
3,xlmr + class weights,0.8993,0.8985,0.9167,0.8293,0.8745,0.9916
4,blend,0.9067,0.9086,0.9243,0.8515,0.8837,0.9905
5,blend + class weights,0.9090,0.9110,0.9261,0.8550,0.8849,0.9930


**Findings.** Blending is a clear gain: 30% TF-IDF + 70% XLM-R scores +0.008 out-of-fold, +0.011 with class weights on
top. The in-sample curve is flat between 20% and 40% TF-IDF, so the exact share does not matter much. The two models
make different errors, mainly on human texts (F1 0.829 → 0.852).

## Experiment 3: stacking with domain and length

In [7]:
# Logistic regression on log-probabilities of both models + domain indicators + number of chunks, test-like weights
def stack_features(split, df, ohe):
    logp = [np.log(np.clip(prob[split][k], 1e-6, 1)) for k in ("xlmr", "xlmr_first", "tfidf")]
    return np.hstack(logp + [np.log(df[["n_chunks"]].to_numpy()), ohe.transform(df[["domain_lang"]])])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(val[["domain_lang"]])
X_val, X_test = stack_features("val", val, ohe), stack_features("test", test, ohe)

def fit_stack(rows):
    return LogisticRegression(C=cfg["stack"]["C"], max_iter=5000).fit(X_val[rows], y[rows], sample_weight=w[rows])

def fit_stack_weights(rows):
    """Stacker on the training folds, then class multipliers on its out-of-fold probabilities there (nested)."""
    inner = np.zeros((len(val), len(LABELS)))
    for k in np.unique(folds[rows]):
        part = rows & (folds != k)
        inner[rows & (folds == k)] = fit_stack(part).predict_proba(X_val[rows & (folds == k)])
    return fit_stack(rows), fit_weights(inner, rows)

oof_stack = out_of_fold(fit_stack, lambda model, rows: model.predict_proba(X_val[rows]))
oof_stack_w = out_of_fold(fit_stack_weights, lambda mm, rows: mm[0].predict_proba(X_val[rows]) * mm[1])
rows.append(report("stacking", oof_stack))
rows.append(report("stacking + class weights", oof_stack_w))
summary = pd.DataFrame(rows)
summary["vs reference"] = summary["macro_f1_testlike"] - summary.loc[summary["variant"] == "reference: xlmr",
                                                                      "macro_f1_testlike"].item()
summary.round(4)

,variant,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai,vs reference
0,tfidf,0.8657,0.8717,0.8917,0.8081,0.8219,0.9850,-0.0328
1,xlmr_first_chunk,0.8599,0.8577,0.8818,0.7743,0.8127,0.9861,-0.0386
2,reference: xlmr,0.8985,0.8975,0.9158,0.8287,0.8739,0.9900,0.0000
3,xlmr + class weights,0.8993,0.8985,0.9167,0.8293,0.8745,0.9916,0.0008
4,blend,0.9067,0.9086,0.9243,0.8515,0.8837,0.9905,0.0083
5,blend + class weights,0.9090,0.9110,0.9261,0.8550,0.8849,0.9930,0.0105
6,stacking,0.9182,0.9202,0.9335,0.8737,0.8919,0.9949,0.0197
7,stacking + class weights,0.9182,0.9202,0.9334,0.8742,0.8918,0.9946,0.0197


**Findings.** Stacking is the best variant: +0.020 over the reference (0.918 out-of-fold), with gains in all three
classes. Class weights on top add nothing: the stacker is already fitted with test-like weights and its probabilities
are calibrated for the decision.

In [8]:
# Macro-F1 per domain (over the classes present there): reference vs the best out-of-fold variant
oof = {"xlmr + class weights": oof_weights, "blend": oof_blend, "blend + class weights": oof_blend_w,
       "stacking": oof_stack, "stacking + class weights": oof_stack_w}
best_name = summary[summary["variant"].isin(oof)].sort_values("macro_f1_testlike").iloc[-1]["variant"]
per_domain = {}
for name, s in (("reference", p_ref), (best_name, oof[best_name])):
    pred = np.array(LABELS)[s.argmax(axis=1)]
    per_domain[name] = {dl: f1_score(g["label"], pred[g.index], labels=sorted(g["label"].unique()), average="macro",
                                     zero_division=0) for dl, g in val.groupby("domain_lang")}
dom = pd.DataFrame(per_domain)
dom["texts"] = val["domain_lang"].value_counts()
dom["change"] = dom[best_name] - dom["reference"]
print("best variant:", best_name)
dom.round(3).sort_values("change")

best variant: stacking


,reference,stacking,texts,change
gutenberg | en-EN,0.786,0.773,692,-0.013
cnn_news | en-EN,0.963,0.967,4429,0.004
student_essays | en-EN,0.922,0.934,5411,0.012
house_of_commons | en-EN,0.899,0.921,3186,0.022
gutenberg | de-DE,0.662,0.685,43,0.023
blog_authorship_corpus | en-EN,0.836,0.861,4053,0.025
bundestag | de-DE,0.891,0.924,5234,0.033
euro_court_cases | en-EN,0.839,0.909,2759,0.070
arxiv_papers | en-EN,0.628,0.740,1429,0.112


**Findings.** The gain of stacking comes mostly from the domains where XLM-R was weakest: arxiv (+0.11) and court cases
(+0.07), then Bundestag, blogs and House of Commons. With the domain as an input the stacker can trust a different
model or shift the decision per domain. Only gutenberg en gets slightly worse (−0.01, 692 texts).

## Test predictions

In [9]:
# Fit every variant on all of validation, predict test, write one submission file per variant (not all are submitted)
all_rows = np.ones(len(val), dtype=bool)
stack_model, m_stack = fit_stack_weights(all_rows)
test_scores = {
    "approach_weights": prob["test"]["xlmr"] * m_full,
    "approach_blend": blend(fit_share(all_rows), "test"),
    "approach_blend_weights": blend(a_full, "test") * m_blend,
    "approach_stack": fit_stack(all_rows).predict_proba(X_test),
    "approach_stack_weights": stack_model.predict_proba(X_test) * m_stack,
}
shares = {}
for name, s in test_scores.items():
    labels = np.array(LABELS)[s.argmax(axis=1)]
    pd.concat([test[["id"]], pd.DataFrame(s / s.sum(axis=1, keepdims=True), columns=COLS)], axis=1).to_csv(
        run.pred_path(name.replace("approach_", "test_")), index=False)
    shares[name] = pd.Series(labels).value_counts(normalize=True)
    make_submission(test["id"], labels, P.repo / "submissions" / f"{name}.csv", P.data)
pd.DataFrame(shares).T[LABELS].round(3)

submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/approach_weights.csv  (25000 rows)
label
ai        0.515
fusion    0.270
human     0.215


submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/approach_blend.csv  (25000 rows)
label
ai        0.519
fusion    0.250
human     0.231


submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/approach_blend_weights.csv  (25000 rows)
label
ai        0.514
fusion    0.247
human     0.239


submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/approach_stack.csv  (25000 rows)
label
ai        0.514
human     0.259
fusion    0.227


submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/approach_stack_weights.csv  (25000 rows)
label
ai        0.513
human     0.264
fusion    0.223


,human,fusion,ai
approach_weights,0.215,0.270,0.515
approach_blend,0.231,0.250,0.519
approach_blend_weights,0.239,0.247,0.514
approach_stack,0.259,0.227,0.514
approach_stack_weights,0.264,0.223,0.513


**Findings.** The predicted class shares on test move towards human as the variants get better (21% → 26%), close
to the validation share of human (25%).

In [10]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)

total: 0.9 min


[]